# 03 Segment-Routed Activation H03

H03 is the final activation-family test before deciding whether to keep moving in this direction.

H01 showed activation-aware treatment helped slow/medium segments but hurt aggregate business metrics. H02 narrowed the training treatment and improved the model read, but still worsened overstock/bias. H03 changes the lever from training-row selection to policy routing: serve activation-aware forecasts only where the signal has been useful, and preserve champion forecasts for protected fast/high-volume series.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "manual_ds":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORT_DIR = PROJECT_ROOT / "backend" / "reports" / "experiments"

REPORT_PATHS = {
    "h01_global_activation": REPORT_DIR / "22aedb84-a49b-499f-a6a4-f0b071c312fd.report.json",
    "h02_segment_gated_activation": REPORT_DIR / "e57a6c4b-a54e-4025-8350-afdb1da7f859.report.json",
    "h03_segment_routed_activation": REPORT_DIR / "0e22686c-b05a-40a0-977c-8977de1f373e.report.json",
}

def load_report(path: Path) -> dict:
    with path.open() as f:
        return json.load(f)

reports = {name: load_report(path) for name, path in REPORT_PATHS.items()}
h03 = reports["h03_segment_routed_activation"]


## Routing Check

Before judging the metrics, I want to confirm H03 actually ran a different policy from H02. The activation challenger should not be served to protected fast movers.


In [ ]:
activation = h03["activation_policy"]
activation_summary = {
    "policy_type": activation["policy_type"],
    "training_rows_excluded": activation["training_rows_excluded"],
    "candidate_pre_activation_training_rows": activation["candidate_pre_activation_training_rows"],
    "protected_pre_activation_training_rows": activation["protected_pre_activation_training_rows"],
    "routed_challenger_holdout_rows": activation["routed_challenger_holdout_rows"],
    "routed_champion_holdout_rows": activation["routed_champion_holdout_rows"],
}
activation_summary


In [ ]:
pd.DataFrame.from_dict(activation["routing_policy_by_segment"], orient="index").sort_index()


## Activation Family Scoreboard

This table compares the three activation experiments against the same champion. The important question is not only whether WAPE moved. The important question is whether forecast quality and inventory decision metrics move together.


In [ ]:
METRICS = [
    "wape",
    "mase",
    "mae",
    "bias_pct",
    "coverage",
    "lost_sales_qty",
    "opportunity_cost_stockout",
    "overstock_rate",
    "overstock_dollars",
    "opportunity_cost_overstock",
]

def pct_delta(base, cand):
    if base in (None, 0) or cand is None:
        return None
    return (cand - base) / base

def primary_scoreboard(reports: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for name, report in reports.items():
        champion = report["baseline"]["holdout_metrics"]
        challenger = report["challenger"]["holdout_metrics"]
        row = {"experiment": name}
        for metric in METRICS:
            base = champion.get(metric)
            cand = challenger.get(metric)
            row[f"{metric}_champion"] = base
            row[f"{metric}_challenger"] = cand
            row[f"{metric}_pct_delta"] = pct_delta(base, cand)
        row["benchmark_gates_passed"] = report["promotion_comparison"]["benchmark_gates_passed"]
        row["failed_gates"] = report["promotion_comparison"]["reason"]
        rows.append(row)
    return pd.DataFrame(rows)

primary_scoreboard(reports)


## H03 Primary Metrics

H03 improved the model metrics and the overstock side of the business tradeoff. It failed because the stockout/lost-sales side got worse.


In [ ]:
def metric_table(report: dict) -> pd.DataFrame:
    champion = report["baseline"]["holdout_metrics"]
    challenger = report["challenger"]["holdout_metrics"]
    rows = []
    for metric in METRICS:
        base = champion.get(metric)
        cand = challenger.get(metric)
        rows.append({
            "metric": metric,
            "champion": base,
            "h03_challenger": cand,
            "absolute_delta": None if base is None or cand is None else cand - base,
            "pct_delta": pct_delta(base, cand),
        })
    return pd.DataFrame(rows)

metric_table(h03)


## Segment Read

H03 did what it was supposed to do at the segment layer. Protected fast/high-volume behavior stayed near champion, while slow and medium improved. That makes this a useful experiment even though it is not a champion.


In [ ]:
def segment_comparison(report: dict) -> pd.DataFrame:
    baseline = report["baseline"].get("segment_metrics") or {}
    challenger = report["challenger"].get("segment_metrics") or {}
    rows = []
    for segment in sorted(set(baseline) | set(challenger)):
        b_payload = baseline.get(segment) or {}
        c_payload = challenger.get(segment) or {}
        b_metrics = b_payload.get("metrics") or {}
        c_metrics = c_payload.get("metrics") or {}
        rows.append({
            "segment": segment,
            "rows": c_payload.get("sample_rows") or b_payload.get("sample_rows"),
            "champion_wape": b_metrics.get("wape"),
            "challenger_wape": c_metrics.get("wape"),
            "wape_delta": None if b_metrics.get("wape") is None or c_metrics.get("wape") is None else c_metrics.get("wape") - b_metrics.get("wape"),
            "champion_bias": b_metrics.get("bias_pct"),
            "challenger_bias": c_metrics.get("bias_pct"),
            "champion_overstock_rate": b_metrics.get("overstock_rate"),
            "challenger_overstock_rate": c_metrics.get("overstock_rate"),
        })
    return pd.DataFrame(rows)

segment_comparison(h03).query("segment in ['fast', 'high_volume', 'medium', 'slow', 'intermittent']")


## Decision Replay

The decision replay is where H03 fails. It reduced overstock dollars, but increased stockout days, lost-sales proxy, lost-sales units, and combined cost. For an inventory decision platform, that is not promotion-worthy.


In [ ]:
DECISION_METRICS = [
    "service_level",
    "stockout_days",
    "lost_sales_proxy",
    "lost_sales_units",
    "overstock_dollars",
    "combined_cost_proxy",
    "po_count",
]

def decision_replay_table(report: dict) -> pd.DataFrame:
    champion = report["decision_replay"]["results"]["champion"]
    challenger = report["decision_replay"]["results"]["challenger"]
    rows = []
    for metric in DECISION_METRICS:
        base = champion.get(metric)
        cand = challenger.get(metric)
        rows.append({
            "metric": metric,
            "champion_policy": base,
            "h03_policy": cand,
            "absolute_delta": None if base is None or cand is None else cand - base,
            "pct_delta": pct_delta(base, cand),
        })
    return pd.DataFrame(rows)

decision_replay_table(h03)


## Senior DS Decision

Stop activation-window/routing as the main hypothesis path.

The activation signal is real and useful: slow and medium segments improved repeatedly, and H03 proved protected fast/high-volume series can be routed back to champion behavior. But the family has not produced a balanced inventory decision improvement.

The pattern is clear:

- H01 and H02 reduced lost-sales pressure but increased overstock/bias.
- H03 reduced overstock and improved aggregate forecast metrics but increased stockout/lost-sales risk.

That means activation is now a documented segment/policy insight, not the next best primary lever. The next manual DS step should go back to EDA and test a new demand driver, starting with price/promotion proxy features from M5 sell-price movement.
